# Stage 1, Gate 1 evidence: 5 seeds for Flat, A and B

The 3-seed control showed A with lower probe forgetting than Flat at all three levels (about 1.4 points at layer3 and
layer4), but within roughly one to two seed standard deviations. The plan asks for 5 seeds before a final claim, so this
adds **seeds 3 and 4** for:

- **Flat** (`flat`, lr 0.1): the reference.
- **A** (`nested_a`, lr 0.5): the candidate.
- **B** (`nested_b`, lr 0.6): Flat + LLRD through the same code path. Gate 1 asks for a gain over Flat *and*
  Flat + LLRD, so B gets the same 5 seeds. Delete it from `ARMS` to save 2 runs.

The configs are identical to the existing control runs except for the seed (same stage `stage1_nested_control`, same lrs
from validation), so seeds 0-2 are recognised as finished and skipped. C and D stay at 3 seeds (no gain to confirm).
Then re-run `stage1_analysis`, `stage1_drift_diagnostic` and `stage1_structural` (CPU): they pick up the new runs.

In [ ]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # <-- EDIT ONCE
BRANCH = "nested-b-d"   # the branch with the code this notebook needs (set back to "main" after merging)

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", url, CODE], check=True)
# fetch + checkout + pull, so a clone left on another branch earlier in the session switches over
subprocess.run(["git", "-C", CODE, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", CODE, "pull", "-q", "origin", BRANCH], check=True)
print(BRANCH, subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
_MINE = {"fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic",
         "stage1_structural", "nested_builder"}
for _m in list(sys.modules):
    if _m in _MINE or _m == "src" or _m.startswith("src."):   # src.* = the team's nested code
        sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = True
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

In [ ]:
import numpy as np, pandas as pd

def table(configs):
    """One row per config: accuracy metrics, cost, and per-level means over tasks (nested arms)."""
    rows = []
    for cfg in configs:
        base = {"arm": sr.arm_label(cfg), "lr": cfg["lr"], "seed": cfg["seed"]}
        r = load_result(cfg, paths)
        if r is None:
            rows.append({**base, "status": "not finished"}); continue
        m = r["metrics"]
        row = {**base, "run_id": r["run_id"], "avg_incremental_acc": m["avg_incremental_acc"],
               "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
               # mean of the diagonal: accuracy on each task right after learning it (plasticity check)
               "acc_just_learned": float(np.mean(np.diag(m["acc_matrix"]))),
               "steps": m["optimizer_steps"], "n_params": m["n_params"],
               "minutes": r["wall_clock_s"] / 60, "peak_gpu_mb": m.get("peak_gpu_mem_mb")}
        log = m.get("level_log_per_exp") or []
        for k in (log[0] if log else {}):
            if k.startswith(("updates/", "rel_displacement/", "deployed_rel_displacement/", "penalty_ratio/")):
                row[k] = float(np.mean([x[k] for x in log]))
        rows.append(row)
    return pd.DataFrame(rows)

## 1. Seeds 3 and 4 (test set, checkpoints)
Estimated ~9 min per run, so ~55 min for 6 new runs (not measured for seeds 3-4).

In [ ]:
EPOCHS = 10
SEEDS = [0, 1, 2, 3, 4]               # 0-2 exist and are skipped
ARMS = {"flat": 0.1, "nested_a": 0.5, "nested_b": 0.6}   # tuned lrs from validation (unchanged)
GATE1 = [sr.arm_cfg(name, stage="stage1_nested_control", epochs=EPOCHS, seed=s, lr=lr,
                    eval_split="test", save_checkpoints=True)
         for s in SEEDS for name, lr in ARMS.items()]   # seed outer: each seed finishes every arm first
for c in GATE1: print(run_id(c), sr.arm_label(c), c["lr"], c["seed"], "| done" if load_result(c, paths) else "")

In [ ]:
out = run_sweep(GATE1, sr.run_experiment, paths)

In [ ]:
cdf = table(GATE1)
done = cdf[cdf["status"] != "not finished"] if "status" in cdf else cdf
assert done["steps"].nunique() == 1 and done["n_params"].nunique() == 1, "steps or param count differ between arms. The comparison is NOT matched"
cols = ["avg_incremental_acc", "acc_just_learned", "final_avg_acc", "avg_forgetting", "minutes"]
display(done.groupby(["arm", "lr"])[cols].agg(["mean", "std", "count"]).round(2))

## 2. Averaging control: every level steps every 4 minibatches (`nested_a_4-4-4`)

Question: is A's lower probe forgetting caused by **different timescales per level** (nesting), or simply by stepping on
gradients **averaged over several minibatches**? This arm is A's own rule with periods (4, 4, 4): every level, head
included, steps every 4 minibatches on the mean of its accumulated gradients. Same minibatches, same backward passes,
no timescale difference between levels.

- Control ≈ A on probe forgetting → A's gain is gradient averaging, not nesting (deflationary result).
- Control clearly worse than A (and new-task accuracy not much lower) → the per-level timescales matter.

Protocol as for every other arm: validation lr grid on seed 0, selection by highest average incremental accuracy,
extend the grid if the best lr is at an edge; then 5 seeds on the test set with checkpoints.

In [ ]:
AVG_GRID = [0.1, 0.3, 0.5, 0.7, 0.9]   # same grid as A
AVG = dict(nesting=dict(periods=[4, 4, 4]))
AVG_VAL = [sr.arm_cfg("nested_a", stage="stage1_nested_lr_grid", epochs=EPOCHS, seed=0, lr=lr, eval_split="val", **AVG)
           for lr in AVG_GRID]
for c in AVG_VAL: print(run_id(c), sr.arm_label(c), c["lr"])
out = run_sweep(AVG_VAL, sr.run_experiment, paths)
g = table(AVG_VAL)
display(g[["arm", "lr", "avg_incremental_acc", "acc_just_learned", "final_avg_acc", "minutes"]])
best = g.loc[g["avg_incremental_acc"].idxmax()]
print(f"best lr: {best['lr']} (avg incremental {best['avg_incremental_acc']:.2f})")
if best["lr"] in (min(AVG_GRID), max(AVG_GRID)): print("WARNING: best lr at the edge of the grid; extend AVG_GRID first.")

In [ ]:
AVG_LR = None                      # <- fill in from the grid above
if AVG_LR is None:
    print("set AVG_LR first")
else:
    AVG_TEST = [sr.arm_cfg("nested_a", stage="stage1_nested_control", epochs=EPOCHS, seed=s, lr=AVG_LR,
                           eval_split="test", save_checkpoints=True, **AVG) for s in SEEDS]
    for c in AVG_TEST: print(run_id(c), sr.arm_label(c), c["lr"], c["seed"])
    out = run_sweep(AVG_TEST, sr.run_experiment, paths)
    cdf = table(GATE1 + AVG_TEST)
    done = cdf[cdf["status"] != "not finished"] if "status" in cdf else cdf
    assert done["steps"].nunique() == 1 and done["n_params"].nunique() == 1, "steps or param count differ between arms. The comparison is NOT matched"
    display(done.groupby(["arm", "lr"])[cols].agg(["mean", "std", "count"]).round(2))
    lvl = [c for c in done.columns if c.startswith(("updates/", "rel_displacement/"))]
    display(done.groupby(["arm", "lr"])[lvl].mean().T.round(4))

**Next:** re-run `stage1_analysis` → `stage1_drift_diagnostic` → `stage1_structural`. The Gate 1 numbers are the
per-level `probe_forgetting` rows (layer3, layer4) with 5 seeds each, plus `acc_just_learned` as the plasticity check.